# Visual Health — RETFound-Green et fine-tuning sur EyePACS

Deux mesures dans une seule session, pour ne télécharger EyePACS qu'une fois :

1. **RETFound-Green** : embeddings de RFMiD, APTOS et EyePACS, comparés
   ensuite à RETFound (`scripts/green_analysis.py`, en local).
2. **Fine-tuning partiel de RETFound** : mêmes réglages que le premier run,
   prédictions sur APTOS et EyePACS. Le test RFMiD n'est **pas** relu.

**Avant de lancer (Colab)** :
1. `Exécution > Modifier le type d'exécution > GPU T4`.
2. Secrets 🔑, accès activé pour ce notebook : `HF_TOKEN` et
   `KAGGLE_API_TOKEN`.
3. `Exécution > Tout exécuter` (~2 h). **Garde l'onglet ouvert.**
4. **Deux téléchargements** : `visual_health_green.zip` (au bout d'~1 h),
   puis `visual_health_finetune_eyepacs.zip` (~200 Mo, avec les poids).
   Si le navigateur demande d'autoriser plusieurs téléchargements, accepte.
5. À la fin : **`Exécution > Déconnecter et supprimer l'environnement
   d'exécution`**.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Pas de GPU : choisir GPU T4"
!df -h /content | tail -1


In [ ]:
import os
WORK = "/content"
%cd {WORK}
![ -d visual-health-platform ] || git clone -q https://github.com/sumarc183-design/visual-health-platform.git
!cd visual-health-platform && git fetch -q && git reset -q --hard origin/main && git log --oneline -1
%cd {WORK}/visual-health-platform
!pip install -q -r requirements.txt kagglehub
!pip install -q -U kaggle
!apt-get -qq install -y p7zip-full > /dev/null


In [ ]:
import glob
from google.colab import userdata

TARGET = "third_party/retfound/checkpoints/RETFound_mae_natureCFP.pth"
if not os.path.exists(TARGET):
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # jamais écrit dans le notebook
    !python scripts/download_retfound_checkpoint.py --repo-id YukunZhou/RETFound_mae_natureCFP --out third_party/retfound/checkpoints
CHECKPOINT = TARGET

GREEN = "third_party/retfound_green/retfoundgreen_statedict.pth"
if not os.path.exists(GREEN):
    !mkdir -p third_party/retfound_green
    !curl -sL --fail -o {GREEN} https://github.com/justinengelmann/RETFound_Green/releases/download/v0.1/retfoundgreen_statedict.pth
GREEN_SHA = "431de5dbc1bebbb32f60e2c0bcf8daa4f8bcbf06f7cb1e1dc97ec589713942e1"
found = !sha256sum {GREEN}
if found[0].split()[0] != GREEN_SHA:
    raise RuntimeError(f"Poids RETFound-Green inattendus : {found[0]}")

token = userdata.get("KAGGLE_API_TOKEN").strip()
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f:
    f.write(token)
os.chmod(os.path.expanduser("~/.kaggle/access_token"), 0o600)
os.environ["KAGGLE_API_TOKEN"] = token
print(CHECKPOINT, GREEN, "OK")


## Données

RFMiD, APTOS, puis les **mêmes 2 500 patients** EyePACS que la validation
externe (graine 0).


In [ ]:
import kagglehub

RFMID_ROOT = kagglehub.dataset_download("andrewmvd/retinal-disease-classification")
print("RFMiD :", RFMID_ROOT)

APTOS_ROOT = f"{WORK}/aptos"
if not os.path.exists(f"{APTOS_ROOT}/train.csv"):
    !kaggle competitions download -c aptos2019-blindness-detection -p {WORK} -q
    !mkdir -p {APTOS_ROOT} && unzip -q -o {WORK}/aptos2019-blindness-detection.zip "train.csv" "train_images/*" -d {APTOS_ROOT}
    !rm {WORK}/aptos2019-blindness-detection.zip
print("APTOS :", APTOS_ROOT, len(os.listdir(f"{APTOS_ROOT}/train_images")), "images")


In [ ]:
import numpy as np
import pandas as pd

ROOT = f"{WORK}/eyepacs"
DL = f"{WORK}/eyepacs_zip"
os.makedirs(ROOT, exist_ok=True)
os.makedirs(DL, exist_ok=True)
COMP = "diabetic-retinopathy-detection"
if not os.path.exists(f"{ROOT}/trainLabels.csv"):
    !kaggle competitions download -c {COMP} -f trainLabels.csv.zip -p {DL} -q
    !unzip -q -o {DL}/trainLabels.csv.zip -d {ROOT}

labels = pd.read_csv(f"{ROOT}/trainLabels.csv")
patients = sorted(labels["image"].str.rsplit("_", n=1).str[0].unique())
chosen = set(np.random.default_rng(0).choice(patients, size=2500, replace=False))
names = labels.loc[labels["image"].str.rsplit("_", n=1).str[0].isin(chosen), "image"]
with open(f"{DL}/list.txt", "w") as f:
    f.write("\n".join(f"train/{n}.jpeg" for n in names))
print(len(chosen), "patients ;", len(names), "images ; stades", labels.set_index("image").loc[names, "level"].value_counts().sort_index().to_dict())

In [ ]:
import subprocess


def sh(cmd: str) -> None:
    """Commande visible ; arrêt immédiat si elle échoue."""
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)


def extracted() -> int:
    return len(glob.glob(f"{ROOT}/train/*.jpeg"))


if extracted() < len(names):
    for part in range(1, 6):
        name = f"train.zip.00{part}"
        if not os.path.exists(f"{DL}/{name}"):
            sh(f"kaggle competitions download -c {COMP} -f {name} -p {DL}")
            if os.path.exists(f"{DL}/{name}.zip"):  # Kaggle livre parfois le fichier dans un zip
                sh(f"unzip -q -o {DL}/{name}.zip -d {DL} && rm -f {DL}/{name}.zip")
        if not os.path.exists(f"{DL}/{name}"):
            raise RuntimeError(f"{name} absent après téléchargement : règlement accepté ? contenu de {DL} : {os.listdir(DL)}")
        print(f"  {name} : {os.path.getsize(f'{DL}/{name}') / 1e9:.1f} Go")
    sh(f"7z l {DL}/train.zip.001 | grep -m 3 -i jpeg || true")  # chemins internes de l'archive
    sh(f"7z x {DL}/train.zip.001 -o{ROOT} -i@{DL}/list.txt -y -bso0 -bsp0")
    if extracted() == 0:  # autre arborescence interne : on cherche les images extraites
        found = glob.glob(f"{ROOT}/**/*.jpeg", recursive=True)
        if found:
            os.makedirs(f"{ROOT}/train", exist_ok=True)
            for f in found:
                os.replace(f, f"{ROOT}/train/{os.path.basename(f)}")
    if extracted() == 0:
        raise RuntimeError("aucune image extraite : archives conservées dans " + DL + " pour diagnostic")
    sh(f"rm -f {DL}/train.zip.00*")
print(extracted(), "images extraites sur", len(names))
!df -h /content | tail -1


## 1. Embeddings RETFound-Green

392 px, `crop-mask`. Le zip se télécharge dès la fin de cette étape : ce
résultat est gardé même si la suite s'interrompt.


In [ ]:
import time
import zipfile

t0 = time.time()
for dataset, root in (("rfmid", RFMID_ROOT), ("aptos", APTOS_ROOT), ("eyepacs", ROOT)):
    !python -u scripts/extract_embeddings.py --dataset {dataset} --backbone green --data-root "{root}" \
        --checkpoint {GREEN} --out-dir embeddings_green --batch-size 64 --num-workers 2 2>&1 | grep -v -i warning
    out = f"embeddings_green/{dataset}_crop-mask_392_retfoundgreen_statedict.npz"
    if not os.path.exists(out) or os.path.getmtime(out) < t0:
        raise RuntimeError(f"extract_embeddings.py a échoué sur {dataset} (voir la sortie ci-dessus)")

archive = f"{WORK}/visual_health_green.zip"
with zipfile.ZipFile(archive, "w") as z:
    for f in sorted(glob.glob("embeddings_green/*.npz")):
        z.write(f)
print("->", archive, f"({os.path.getsize(archive) / 1e6:.0f} Mo) en {(time.time() - t0) / 60:.0f} min")
from google.colab import files
files.download(archive)


## 2. Fine-tuning partiel, puis APTOS et EyePACS

Mêmes réglages que le premier run. Les poids sont enregistrés avant les bases
externes. Si une base échoue, garde la session ouverte : on relancera avec
`--external-only`, sans réentraîner.


In [ ]:
t0 = time.time()
!python -u scripts/finetune_retfound.py --data-root "{RFMID_ROOT}" --checkpoint {CHECKPOINT} \
    --out-dir finetune_eyepacs --aptos-root "{APTOS_ROOT}" --eyepacs-root "{ROOT}" --workers 2 2>&1 | grep -v -i warning
for f in ("finetune_rfmid.pt", "pred_aptos_finetune.npz", "pred_eyepacs_finetune.npz"):
    path = f"finetune_eyepacs/{f}"
    if not os.path.exists(path) or os.path.getmtime(path) < t0:
        raise RuntimeError(f"finetune_retfound.py a échoué : {path} absent (voir la sortie ci-dessus)")


## Garder le résultat

Prédictions, historique et **poids entraînés**.


In [ ]:
archive = f"{WORK}/visual_health_finetune_eyepacs.zip"
with zipfile.ZipFile(archive, "w") as z:
    for f in sorted(glob.glob("finetune_eyepacs/*")):
        z.write(f)
print("->", archive, f"({os.path.getsize(archive) / 1e6:.0f} Mo)")
files.download(archive)
